# Stage 1a — rate CFD with an open-weight VLM

Scores every neutral CFD image with one model, zero-shot, using the ordinal
expected-rating task score.

**Before you start**
1. Runtime → Change runtime type → **L4 GPU** (Colab Pro). The free T4 has 16 GB;
   a 7B VLM at fp16 needs ~17 GB of weights alone. Cell 1 enforces this.
2. Have the unzipped `CFD Version 3.0` folder in your Drive.
3. Set `REPO_URL` in cell 2.

**Do not quantize to fit a smaller GPU.** The dependent variable is the logit
distribution over the seven rating tokens; quantization perturbs exactly that,
and at Stage 4 it contaminates the gradients the CAV sensitivity is built from.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{name}  |  {vram:.1f} GB")

if vram < 20:
    raise RuntimeError(
        f"{name} has only {vram:.1f} GB. A 7B VLM at fp16 needs ~17 GB of weights "
        "alone. Switch to an L4 (24 GB) or A100. Do not work around this by "
        "quantizing -- see the note at the top."
    )

## 2. Get the code

In [ ]:
REPO_URL = "https://github.com/YOURUSER/face.git"  # <-- set this

import os
import subprocess

if not os.path.exists("/content/face"):
    subprocess.run(["git", "clone", REPO_URL, "/content/face"], check=True)

os.chdir("/content/face")
!git log --oneline -1

## 3. Install dependencies

Editable install so `facecav` imports from the scripts. Torch is intentionally
not reinstalled — Colab's build is CUDA-matched and replacing it breaks CUDA.

In [ ]:
!pip install -q -e ".[dev]"

## 4. Point at the CFD images in your Drive

Set `DRIVE_CFD` to your unzipped `CFD Version 3.0` folder. No zipping needed.

`COPY_LOCAL = True` copies it to local disk once (~1–2 min). Reading the 831
JPEGs straight off Drive's FUSE layer does work, but it adds per-file latency to
every run and intermittently throws I/O errors under repeated access. Set it to
`False` to read in place.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CFD = "/content/drive/MyDrive/CFD Version 3.0"   # <-- adjust to yours
COPY_LOCAL = True

import shutil
from pathlib import Path

WORKBOOK = "CFD 3.0 Norming Data and Codebook.xlsx"
drive_root = Path(DRIVE_CFD)

if not (drive_root / WORKBOOK).exists():
    print(f"Not found: {drive_root / WORKBOOK}\nSearching Drive...")
    hits = list(Path("/content/drive/MyDrive").glob(f"**/{WORKBOOK}"))
    raise SystemExit(
        "Set DRIVE_CFD to one of:\n  " + "\n  ".join(str(h.parent) for h in hits)
        if hits
        else "Could not find the CFD workbook anywhere in MyDrive."
    )

if COPY_LOCAL:
    CFD_ROOT = Path("/content/cfd/CFD Version 3.0")
    if not (CFD_ROOT / WORKBOOK).exists():
        CFD_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive_root, CFD_ROOT, dirs_exist_ok=True)
else:
    CFD_ROOT = drive_root

CFD_ROOT_STR = str(CFD_ROOT)
print(f"{CFD_ROOT}\n{sum(1 for _ in CFD_ROOT.rglob('*-N.jpg'))} neutral images (expect 831)")

## 5. Verify the pipeline before spending GPU time

Runs the test suite and builds the manifest — no GPU, no model download.
Expect **25 passing** and **831 rows / 826 matched**. If this fails, stop:
nothing downstream can be right.

In [ ]:
!python -m pytest tests/ -q

from facecav.data.cfd import build_manifest

manifest = build_manifest(CFD_ROOT)
print(f"\nrows: {len(manifest)}   matched: {(manifest.join_status == 'matched').sum()}")
print(manifest.join_status.value_counts().to_string())

## 6. Smoke test — three images

The first real forward pass, and where the untested assumptions in `rater.py`
surface: whether the chat template renders, whether the processor accepts the
image, and whether the logits land on the position after `"The rating is "`.

**What to check, worst first:**
- **Identical ratings across all three faces** → the image is not reaching the
  model, and every downstream number would be an artifact of the prompt alone.
- **`refusal_mass` near 1.0** → the model is not answering with a digit.
- **Rating outside [1, 7]** → impossible by construction; token IDs resolved wrong.

These are the first three manifest rows (all Asian female), so read the wiring,
not the values.

In [ ]:
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

!python experiments/stage1a_rate_cfd.py --model "$MODEL" --cfd-root "$CFD_ROOT_STR" --limit 3

import json
from pathlib import Path

out = Path("artifacts/stage1a") / f"{MODEL.replace('/', '__')}.jsonl"
for line in out.read_text().splitlines():
    r = json.loads(line)
    print(f"{r['model_id']:>12}  rating={r['expected_rating']:.3f}  "
          f"refusal={r['refusal_mass']:.4f}  probs={[round(p, 3) for p in r['rating_probs']]}")

## 7. Full Stage 1a

All 831 images. Resumable: results append to JSONL and completed images are
skipped, so a disconnect costs at most one image — just rerun this cell.

In [ ]:
!python experiments/stage1a_rate_cfd.py --model "$MODEL" --cfd-root "$CFD_ROOT_STR"

## 8. First look

In [ ]:
import pandas as pd

ratings = pd.read_json(out, lines=True)
print(f"n = {len(ratings)}")
print(f"\nrefusal mass: mean={ratings.refusal_mass.mean():.4f}  max={ratings.refusal_mass.max():.4f}")

print("\nmean expected rating by race x gender:")
print(pd.crosstab(ratings.race_code, ratings.gender_code,
                  values=ratings.expected_rating, aggfunc="mean").round(3).to_string())

print("\nrefusal by race (differential refusal is itself a finding -- spec 5.6):")
print(ratings.groupby("race_code").refusal_mass.mean().round(4).to_string())

## 9. Save results back to Drive

Colab storage is ephemeral. The JSONL is small — always copy it out.

In [ ]:
!mkdir -p "/content/drive/MyDrive/face_artifacts"
!cp -r artifacts/stage1a "/content/drive/MyDrive/face_artifacts/"
!ls -la "/content/drive/MyDrive/face_artifacts/stage1a"

---
### Next

Repeat cell 7 for `OpenGVLab/InternVL3-8B` and
`HuggingFaceM4/Idefics3-8B-Llama3`. The 32B/38B arm needs an A100 80GB
(High-RAM toggle).

**Cell 8 is not a result yet.** Spec §9 requires prompt-paraphrase stability and
refusal rates first — a mean-rating table looks publishable while resting on a
single untested prompt. The ICL condition does not exist yet; wiring it needs
the §14 decision on demonstration set size and composition.